# 2_load_eGFR_model — RECOVERY version

커널이 죽어서 메모리가 초기화된 상황을 복구하는 노트북.

**전략**
- 모델·imputer·metadata는 디스크에 저장되어 있음 (변경 없음)
- `RANDOM_STATE=42` 고정이라 `train_test_split`은 결정론적으로 재현됨
- 따라서 **원본 학습 당시와 bit-for-bit 동일한** `y_test_A`, `pred_test_A` 등을 복원 가능
- 이후 `res_A_test` 등 eval 결과도 원본과 동일

**실행 순서**: 위에서 아래로 셀을 순서대로 실행하면 됨.


## Cell 1 — 기존 `load_package()` / `predict()` 헬퍼 (변경 없음)

In [1]:
"""
load_ckd_models.py  (v4-2 기준)  — 기존 2_load_eGFR_model.ipynb와 동일
"""
import argparse, json, sys, os, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.pipeline import Pipeline
from tabpfn.model_loading import load_fitted_tabpfn_model


def load_package(pkg_dir, map_to_cpu: bool = False) -> dict:
    pkg = Path(pkg_dir)

    # ── Step 1. TwoTierImputer 클래스 등록 ───────────────────────────────────
    src_dir = str(pkg / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    try:
        import two_tier_imputer as _tti
        import __main__
        __main__.TwoTierImputer = _tti.TwoTierImputer
        print(f"[LOAD] TwoTierImputer: {_tti.__file__} ✅")
    except ImportError as e:
        raise ImportError(
            f"two_tier_imputer.py를 찾을 수 없습니다.\n"
            f"  탐색 경로: {src_dir}\n"
            f"  오류: {e}"
        ) from e

    # ── Step 2. metadata ─────────────────────────────────────────────────────
    with open(pkg / "metadata.json", encoding="utf-8") as f:
        metadata = json.load(f)
    model_order = metadata["model_order"]
    print(f"[LOAD] metadata: {len(model_order)} models | "
          f"horizon={metadata.get('horizon_days')} days")

    # ── Step 3. 모델 로딩 ────────────────────────────────────────────────────
    result = {"models_A": {}, "models_B": {}, "metadata": metadata}

    for scen_tag, key in [("scenario_A", "models_A"), ("scenario_B", "models_B")]:
        scen_dir = pkg / scen_tag
        if not scen_dir.exists():
            print(f"[LOAD] ⚠️  {scen_tag} 폴더 없음 — 건너뜁니다")
            continue

        for name in model_order:
            pre_path = scen_dir / "preprocess" / f"{name}_preprocess.joblib"
            tpf_path = scen_dir / "tabpfn_fit"  / f"{name}.tabpfn_fit"
            if not pre_path.exists() or not tpf_path.exists():
                print(f"[LOAD] ⚠️  {scen_tag}/{name} 파일 없음 — 건너뜁니다")
                continue

            pre = joblib.load(pre_path)
            clf = load_fitted_tabpfn_model(tpf_path)
            if map_to_cpu and hasattr(clf, "to"):
                clf = clf.to("cpu")

            result[key][name] = Pipeline(steps=[("preprocess", pre), ("clf", clf)])
            print(f"[LOAD] ✅ {scen_tag}/{name}")

        imp_path = scen_dir / "imputer.joblib"
        if imp_path.exists():
            imp     = joblib.load(imp_path)
            imp_key = "imputer_A" if scen_tag == "scenario_A" else "imputer_B"
            result[imp_key] = imp
            print(f"[LOAD] ✅ {scen_tag}/imputer  (include_cap={imp.include_cap})")

    return result


print("[CELL 1] load_package() defined.")


[CELL 1] load_package() defined.


## Cell 2 — 패키지 로드 & config 복원

`metadata.json`에 저장된 모든 상수(`RANDOM_STATE`, `MODEL_SPECS`, `CAP_CUTOFF_BY_HOSPITAL` 등)를 복원합니다.
환경에 맞게 `PKG_DIR`과 `DATA_PATH`만 수정하세요.

In [2]:
# ── 경로 설정 — 환경에 맞게 수정 ────────────────────────────────────────────
PKG_DIR   = "/home/ubuntu/eGFR25_output/models"
DATA_PATH = "/home/ubuntu/Fibroscan_CKD_2025.xlsx"
OUTDIR    = Path("/home/ubuntu/eGFR25_output")

# ── 로드 ─────────────────────────────────────────────────────────────────────
pkg = load_package(PKG_DIR, map_to_cpu=not torch.cuda.is_available())

models_A = pkg["models_A"]
models_B = pkg["models_B"]
imp_A    = pkg["imputer_A"]
imp_B    = pkg["imputer_B"]
metadata = pkg["metadata"]

# ── Config 복원 (metadata.json → 네임스페이스) ──────────────────────────────
MODEL_SPECS    = metadata["model_specs"]
MODEL_ORDER    = metadata["model_order"]
TIER1_FEATURES = metadata["tier1_features"]
TIER2_FEATURES = metadata["tier2_features"]
CAP_CUTOFF_BY_HOSPITAL = {
    int(k): pd.Timestamp(v) for k, v in metadata["cap_cutoff_by_hospital"].items()
}
MICE_AUX_COLS = metadata["mice_aux_cols"]
RANDOM_STATE  = metadata["random_state"]
N_BOOT        = metadata["n_boot"]
HORIZON_DAYS  = metadata["horizon_days"]
OUTCOME       = metadata["outcome"]

# 원본 노트북 Cell 5의 고정값
FIVE_YEARS_DAYS        = HORIZON_DAYS
MIN_FU_DAYS_FOR_COHORT = 365
MAX_TABPFN_N           = 10_000
HOSP_INTERNAL          = 10
HOSP_EXTERNAL          = 20

ALL_MODEL_FEATURES = sorted(set(sum(MODEL_SPECS.values(), [])))

# event / tte 컬럼명
if OUTCOME == "eGFR25":
    event_col, tte_col = "eGFR25", "TTO_eGFR25"
else:
    event_col, tte_col = "CKD", "TTO_CKD"

# two_tier_imputer 모듈 헬퍼들 (load_package가 이미 sys.path에 등록함)
from two_tier_imputer import (
    TwoTierImputer, add_fibroscan_date_numeric, is_cap_era,
)

print(f"\n[CONFIG] Restored from metadata.json ✅")
print(f"  RANDOM_STATE = {RANDOM_STATE}")
print(f"  OUTCOME      = {OUTCOME}  (event_col={event_col}, tte_col={tte_col})")
print(f"  HORIZON_DAYS = {HORIZON_DAYS}")
print(f"  N_BOOT       = {N_BOOT}")
print(f"  MODEL_ORDER  = {MODEL_ORDER}")
print(f"  CAP_CUTOFF   = " + ", ".join(f"H{k}:{v.date()}" for k,v in CAP_CUTOFF_BY_HOSPITAL.items()))


[LOAD] TwoTierImputer: /home/ubuntu/eGFR25_output/models/src/two_tier_imputer.py ✅
[LOAD] metadata: 6 models | horizon=1826 days
[LOAD] ✅ scenario_A/base
[LOAD] ✅ scenario_A/base_lsm
[LOAD] ✅ scenario_A/base_lsm_cap
[LOAD] ✅ scenario_A/base_inflam
[LOAD] ✅ scenario_A/base_inflam_lsm
[LOAD] ✅ scenario_A/base_inflam_lsm_cap
[LOAD] ✅ scenario_A/imputer  (include_cap=False)
[LOAD] ✅ scenario_B/base
[LOAD] ✅ scenario_B/base_lsm
[LOAD] ✅ scenario_B/base_lsm_cap
[LOAD] ✅ scenario_B/base_inflam
[LOAD] ✅ scenario_B/base_inflam_lsm
[LOAD] ✅ scenario_B/base_inflam_lsm_cap
[LOAD] ✅ scenario_B/imputer  (include_cap=True)

[CONFIG] Restored from metadata.json ✅
  RANDOM_STATE = 42
  OUTCOME      = eGFR25  (event_col=eGFR25, tte_col=TTO_eGFR25)
  HORIZON_DAYS = 1826
  N_BOOT       = 1000
  MODEL_ORDER  = ['base', 'base_lsm', 'base_lsm_cap', 'base_inflam', 'base_inflam_lsm', 'base_inflam_lsm_cap']
  CAP_CUTOFF   = H10:2012-03-14, H20:2013-01-10


## Cell 3 — 원본 엑셀 로드 + cohort exclusion + outcome 생성

**원본 노트북 Cell 3, 11을 그대로 재현.**
결과물: `df_final` — exclusion과 5-yr outcome이 적용된 최종 cohort.

In [3]:
def load_df_raw(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"[LOAD] File not found: {path}")
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(path)
    elif ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    else:
        raise ValueError(f"[LOAD] Unsupported extension: {ext}")
    print(f"[LOAD] df_raw loaded: shape={df.shape}")
    return df


def apply_cohort_exclusions(df: pd.DataFrame) -> pd.DataFrame:
    """원본 Cell 11과 동일"""
    out = df.copy()
    for col in ["Baseline_eGFR","ESRD","KT","Baseline_PU","Malignancy","Follow-up_duration"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    print(f"[EXCLUDE] Start n={len(out):,}")
    mask = pd.Series(True, index=out.index)
    for col, crit, label in [
        ("Baseline_eGFR",       lambda d: d < 60, "Baseline CKD (eGFR<60)"),
        ("ESRD",                lambda d: d == 1, "ESRD==1"),
        ("KT",                  lambda d: d == 1, "KT==1"),
        ("Baseline_PU",         lambda d: d == 1, "Baseline_PU==1"),
        ("Malignancy",          lambda d: d == 1, "Malignancy==1"),
        ("Follow-up_duration",  lambda d: d < MIN_FU_DAYS_FOR_COHORT, "FU<1Y"),
    ]:
        if col in out.columns:
            n_before = mask.sum()
            mask &= ~crit(out[col])
            print(f"[EXCLUDE] {label}: removed {n_before - mask.sum():,}")
    out = out.loc[mask].reset_index(drop=True)
    print(f"[EXCLUDE] End n={len(out):,}")
    return out


def build_5yr_outcome(
    df: pd.DataFrame, event_col: str, tte_col: str,
    fu_col: str = "Follow-up_duration", five_year_days: int = None,
) -> pd.DataFrame:
    """원본 Cell 11과 동일"""
    if five_year_days is None:
        five_year_days = FIVE_YEARS_DAYS
    out = df.copy()
    for col in [event_col, tte_col, fu_col]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    print(f"[OUTCOME] Building y for {event_col} within 5y")
    n0   = len(out)
    cand = out.loc[out[fu_col].notna() & ((out[event_col]!=1) | out[tte_col].notna())].copy()
    print(f"[OUTCOME] After drop missing FU/TTE: {len(cand):,} (removed {n0-len(cand):,})")
    y = pd.Series(np.nan, index=cand.index)
    y.loc[(cand[event_col]==1)&(cand[tte_col]>0)&(cand[tte_col]<=five_year_days)] = 1
    y.loc[(cand[event_col]==0)&(cand[fu_col]>=five_year_days)]                    = 0
    y.loc[(cand[event_col]==1)&(cand[tte_col]>five_year_days)&(cand[fu_col]>=five_year_days)] = 0
    before = len(cand)
    cand   = cand.loc[y.notna()].copy()
    cand["y"] = y.loc[cand.index].astype(int)
    print(f"[OUTCOME] Removed uncertain: {before-len(cand):,}")
    print(f"[OUTCOME] Final n={len(cand):,}, event rate={cand['y'].mean():.3f}")
    return cand.reset_index(drop=True)


# ── Execute ─────────────────────────────────────────────────────────────────
df_raw   = load_df_raw(DATA_PATH)
df_excl  = apply_cohort_exclusions(df_raw)
df_final = build_5yr_outcome(df_excl, event_col=event_col, tte_col=tte_col)

print("\n[FINAL COHORT] by Hospital")
print(df_final.groupby("Hospital")["y"].agg(N="count", Events="sum", EventRate="mean"))


[LOAD] df_raw loaded: shape=(59161, 60)
[EXCLUDE] Start n=59,161
[EXCLUDE] Baseline CKD (eGFR<60): removed 2,139
[EXCLUDE] ESRD==1: removed 87
[EXCLUDE] KT==1: removed 122
[EXCLUDE] Baseline_PU==1: removed 8,213
[EXCLUDE] Malignancy==1: removed 7,451
[EXCLUDE] FU<1Y: removed 4,363
[EXCLUDE] End n=36,786
[OUTCOME] Building y for eGFR25 within 5y
[OUTCOME] After drop missing FU/TTE: 27,713 (removed 9,073)
[OUTCOME] Removed uncertain: 14,232
[OUTCOME] Final n=13,481, event rate=0.188

[FINAL COHORT] by Hospital
             N  Events  EventRate
Hospital                         
10        9945    1881   0.189140
20        3536     649   0.183541


## Cell 4 — Train/Test/Ext split (원본과 동일하게 재현)

**원본 Cell 13 재현.** `RANDOM_STATE=42` 고정이라 결정론적으로 동일한 split을 얻습니다.
이어서 Cell 17의 numeric coercion + `add_fibroscan_date_numeric()`까지 처리.

In [4]:
from sklearn.model_selection import train_test_split

internal = df_final[df_final["Hospital"] == HOSP_INTERNAL].copy()
external = df_final[df_final["Hospital"] == HOSP_EXTERNAL].copy()

idx_train, idx_test = train_test_split(
    internal.index.values, test_size=0.2,
    random_state=RANDOM_STATE, stratify=internal["y"].values
)
df_train = internal.loc[idx_train].copy()
df_test  = internal.loc[idx_test].copy()
df_ext   = external.copy()

# ── Numeric coercion (원본 Cell 17) ─────────────────────────────────────────
for c in ALL_MODEL_FEATURES:
    for _df in [df_train, df_test, df_ext]:
        if c in _df.columns:
            _df.loc[:, c] = pd.to_numeric(_df[c], errors="coerce")

# ── Fibroscan_Date_numeric 추가 (원본 Cell 17) ──────────────────────────────
df_train = add_fibroscan_date_numeric(df_train)
df_test  = add_fibroscan_date_numeric(df_test)
df_ext   = add_fibroscan_date_numeric(df_ext)

# ── Scenario B용 labels (full cohort) ───────────────────────────────────────
y_train = df_train["y"].astype(int).values
y_test  = df_test["y"].astype(int).values   # == y_test_B
y_ext   = df_ext["y"].astype(int).values    # == y_ext_B

print(f"[SPLIT] Internal n={len(internal):,}  External n={len(external):,}")
print(f"[SPLIT] Train n={len(y_train):,} (event={y_train.mean():.3f})")
print(f"[SPLIT] Test  n={len(y_test):,} (event={y_test.mean():.3f})")
print(f"[SPLIT] Ext   n={len(y_ext):,} (event={y_ext.mean():.3f})")


[SPLIT] Internal n=9,945  External n=3,536
[SPLIT] Train n=7,956 (event=0.189)
[SPLIT] Test  n=1,989 (event=0.189)
[SPLIT] Ext   n=3,536 (event=0.184)


## Cell 5 — Scenario A 서브셋 (CAP-era mask)

**원본 Cell 17의 Scenario A 부분 재현.** 병원별 CAP 도입 cutoff 이후의 샘플만 남깁니다.

In [5]:
cap_mask_test = is_cap_era(df_test)
cap_mask_ext  = is_cap_era(df_ext)

df_test_A = df_test[cap_mask_test].copy()
df_ext_A  = df_ext[cap_mask_ext].copy()

y_test_A = df_test_A["y"].astype(int).values
y_ext_A  = df_ext_A["y"].astype(int).values

print(f"[SCENARIO A] test n={len(df_test_A):,}  event={y_test_A.mean():.3f}")
print(f"[SCENARIO A] ext  n={len(df_ext_A):,}  event={y_ext_A.mean():.3f}")
print(f"[SCENARIO B] test n={len(df_test):,}   event={y_test.mean():.3f}  (full cohort)")
print(f"[SCENARIO B] ext  n={len(df_ext):,}    event={y_ext.mean():.3f}  (full cohort)")


[SCENARIO A] test n=1,110  event=0.270
[SCENARIO A] ext  n=1,771  event=0.166
[SCENARIO B] test n=1,989   event=0.189  (full cohort)
[SCENARIO B] ext  n=3,536    event=0.184  (full cohort)


## Cell 6 — 저장된 imputer로 transform

**원본 Cell 18 재현.** 저장된 `imp_A`, `imp_B`의 `transform()`만 호출합니다(fit은 이미 되어 있음).

In [6]:
# ── Scenario A — imp_A는 CAP-era training subset에 fit된 상태 ───────────────
df_test_A_imp = imp_A.transform(df_test_A)
df_ext_A_imp  = imp_A.transform(df_ext_A)

# ── Scenario B — imp_B는 full cohort training에 fit, CAP MICE 포함 ─────────
df_test_B_imp = imp_B.transform(df_test)
df_ext_B_imp  = imp_B.transform(df_ext)

# ── Post-imputation sanity check ───────────────────────────────────────────
print("[IMPUTE] Post-imputation missing check (model features only):")
check_map = [
    ("A test", df_test_A_imp), ("A ext", df_ext_A_imp),
    ("B test", df_test_B_imp), ("B ext", df_ext_B_imp),
]
for label, df_imp in check_map:
    still = {c: int(df_imp[c].isna().sum()) for c in ALL_MODEL_FEATURES
             if c in df_imp.columns and df_imp[c].isna().any()}
    if still:
        print(f"  {label}: ⚠️  residual missing — {still}")
    else:
        print(f"  {label}: ✅  no missing in model features")


[IMPUTE] Post-imputation missing check (model features only):
  A test: ⚠️  residual missing — {'CAP': 27}
  A ext: ⚠️  residual missing — {'CAP': 219}
  B test: ✅  no missing in model features
  B ext: ✅  no missing in model features


## Cell 7 — 예측 딕셔너리 재생성 (`pred_test_A`, `pred_ext_A`, `pred_test_B`, `pred_ext_B`)

**원본 Cell 23-24의 예측 부분만 재현.** GridSearchCV 재학습은 이미 저장된 best_estimator를 쓰므로 스킵.

In [7]:
pred_test_A, pred_ext_A = {}, {}
pred_test_B, pred_ext_B = {}, {}

for name in MODEL_ORDER:
    feats = MODEL_SPECS[name]

    # Scenario A
    pred_test_A[name] = models_A[name].predict_proba(df_test_A_imp[feats])[:, 1]
    pred_ext_A[name]  = models_A[name].predict_proba(df_ext_A_imp[feats])[:, 1]

    # Scenario B
    pred_test_B[name] = models_B[name].predict_proba(df_test_B_imp[feats])[:, 1]
    pred_ext_B[name]  = models_B[name].predict_proba(df_ext_B_imp[feats])[:, 1]

    print(f"  [PRED] {name:22s}  "
          f"A.test={pred_test_A[name].mean():.4f}  A.ext={pred_ext_A[name].mean():.4f}  "
          f"B.test={pred_test_B[name].mean():.4f}  B.ext={pred_ext_B[name].mean():.4f}")

print("\n[PRED] All predictions regenerated ✅")


  [PRED] base                    A.test=0.2501  A.ext=0.2610  B.test=0.1669  B.ext=0.1719
  [PRED] base_lsm                A.test=0.2526  A.ext=0.2475  B.test=0.1681  B.ext=0.1631
  [PRED] base_lsm_cap            A.test=0.2524  A.ext=0.2472  B.test=0.1705  B.ext=0.1633
  [PRED] base_inflam             A.test=0.2488  A.ext=0.2539  B.test=0.1669  B.ext=0.1663
  [PRED] base_inflam_lsm         A.test=0.2515  A.ext=0.2465  B.test=0.1695  B.ext=0.1624
  [PRED] base_inflam_lsm_cap     A.test=0.2513  A.ext=0.2451  B.test=0.1724  B.ext=0.1618

[PRED] All predictions regenerated ✅


## Cell 8 — 평가 함수 복원 (원본 Cell 26 전체 복사)

`evaluate_models_full`을 비롯한 모든 helper들(`disc_metrics`, `cal_metrics`, `nri_*`, `boot_cindex`, `build_surv` 등)을 다시 정의.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

EPS = 1e-12
def _clip(p): return np.clip(p, EPS, 1-EPS)
def _logit(p): p=_clip(p); return np.log(p/(1-p))

def bh_fdr(pvals):
    pvals=np.asarray(pvals,dtype=float); n=len(pvals)
    if n==0: return pvals
    order=np.argsort(pvals)
    q=pvals[order]*n/(np.arange(n)+1)
    q=np.minimum.accumulate(q[::-1])[::-1]
    out=np.empty_like(q); out[order]=np.clip(q,0,1)
    return out

def stratified_bootstrap_indices(y, rng, n_boot=500):
    y=np.asarray(y).astype(int)
    i0,i1=np.where(y==0)[0],np.where(y==1)[0]
    for _ in range(n_boot):
        idx=np.concatenate([rng.choice(i0,len(i0),replace=True),
                             rng.choice(i1,len(i1),replace=True)])
        rng.shuffle(idx); yield idx

def ci_from_boot(d,alpha=0.05):
    d=np.asarray(d,dtype=float)
    if d.size==0: return np.nan,np.nan
    return float(np.quantile(d,alpha/2)),float(np.quantile(d,1-alpha/2))

def two_sided_p(d):
    d=np.asarray(d,dtype=float)
    if d.size==0: return np.nan
    return float(min(1.0,2*min(np.mean(d<=0),np.mean(d>=0))))

def disc_metrics(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p,dtype=float)
    if len(np.unique(y))<2: return {"AUROC":np.nan,"AUPRC":np.nan}
    return {"AUROC":float(roc_auc_score(y,p)),
            "AUPRC":float(average_precision_score(y,p))}

def boot_delta_disc(y,pa,pb,n_boot=500,seed=42):
    y=np.asarray(y).astype(int); pa=np.asarray(pa,dtype=float); pb=np.asarray(pb,dtype=float)
    rng=np.random.default_rng(seed); da,dp=[],[]
    for idx in stratified_bootstrap_indices(y,rng,n_boot):
        yb=y[idx]
        if len(np.unique(yb))<2: continue
        da.append(roc_auc_score(yb,pb[idx])-roc_auc_score(yb,pa[idx]))
        dp.append(average_precision_score(yb,pb[idx])-average_precision_score(yb,pa[idx]))
    return np.asarray(da,dtype=float),np.asarray(dp,dtype=float)

def cal_metrics(y,p):
    y=np.asarray(y).astype(int); p=np.asarray(p,dtype=float)
    lr=LogisticRegression(penalty=None,solver="lbfgs",max_iter=2000)
    lr.fit(_logit(p).reshape(-1,1),y)
    return {"Brier":float(brier_score_loss(y,p)),
            "Cal_Intercept":float(lr.intercept_[0]),
            "Cal_Slope":float(lr.coef_[0][0])}

def nri_cat(y,po,pn,cp=(0.05,0.10,0.20)):
    y=np.asarray(y).astype(int); bins=np.array([0.,*cp,1.+1e-9])
    co=np.digitize(np.asarray(po,dtype=float),bins)-1
    cn=np.digitize(np.asarray(pn,dtype=float),bins)-1
    up,dn=cn>co,cn<co; ev,ne=y==1,y==0
    return float((np.mean(up[ev])-np.mean(dn[ev]))+(np.mean(dn[ne])-np.mean(up[ne])))

def nri_cont(y,po,pn):
    y=np.asarray(y).astype(int)
    d=np.asarray(pn,dtype=float)-np.asarray(po,dtype=float)
    up,dn=d>0,d<0; ev,ne=y==1,y==0
    return float((np.mean(up[ev])-np.mean(dn[ev]))+(np.mean(dn[ne])-np.mean(up[ne])))

def boot_nri(y,po,pn,n_boot=500,seed=42,cp=(0.05,0.10,0.20)):
    y=np.asarray(y).astype(int)
    po=np.asarray(po,dtype=float); pn=np.asarray(pn,dtype=float)
    rng=np.random.default_rng(seed); dc,dk=[],[]
    for idx in stratified_bootstrap_indices(y,rng,n_boot):
        dc.append(nri_cat(y[idx],po[idx],pn[idx],cp))
        dk.append(nri_cont(y[idx],po[idx],pn[idx]))
    return np.asarray(dc,dtype=float),np.asarray(dk,dtype=float)

def harrell_c(t,e,r):
    t=np.asarray(t,dtype=float); e=np.asarray(e,dtype=int); r=np.asarray(r,dtype=float)
    conc=perm=ties=0.0; n=len(t)
    for i in range(n):
        for j in range(i+1,n):
            if e[i]==1 and t[i]<t[j]:
                perm+=1
                if r[i]>r[j]: conc+=1
                elif r[i]==r[j]: ties+=1
            elif e[j]==1 and t[j]<t[i]:
                perm+=1
                if r[j]>r[i]: conc+=1
                elif r[i]==r[j]: ties+=1
    return np.nan if perm==0 else float((conc+0.5*ties)/perm)

def boot_cindex(t,e,r,n_boot=700,seed=42):
    rng=np.random.default_rng(seed)
    dist=[harrell_c(t[idx],e[idx],r[idx])
          for idx in stratified_bootstrap_indices(np.asarray(e).astype(int),rng,n_boot)]
    dist=[x for x in dist if np.isfinite(x)]
    lo,hi=ci_from_boot(np.asarray(dist))
    return harrell_c(t,e,r),lo,hi

def boot_delta_cindex(t,e,pa,pb,n_boot=500,seed=42):
    rng=np.random.default_rng(seed); dist=[]
    for idx in stratified_bootstrap_indices(np.asarray(e).astype(int),rng,n_boot):
        ca=harrell_c(t[idx],e[idx],pa[idx]); cb=harrell_c(t[idx],e[idx],pb[idx])
        if np.isfinite(ca) and np.isfinite(cb): dist.append(cb-ca)
    return np.asarray(dist,dtype=float)

def build_surv(df,event_col,tte_col,fu_col,horizon):
    ev=pd.to_numeric(df[event_col],errors="coerce").values
    tte=pd.to_numeric(df[tte_col],errors="coerce").values
    fu=pd.to_numeric(df[fu_col],errors="coerce").values
    t=np.minimum(fu,horizon).astype(float)
    e=np.zeros(len(df),dtype=int)
    m=(ev==1)&(tte>0)&(tte<=horizon)&np.isfinite(tte)
    e[m]=1; t[m]=tte[m]
    return t,e


def evaluate_models_full(
    y_true, pred_proba, df_split,
    event_col, tte_col, fu_col="Follow-up_duration",
    horizon_days=1826, n_boot=500, seed=42
):
    names=list(pred_proba.keys())
    print(f"[EVAL] models={names}")

    summary=pd.DataFrame(
        [{"Model":m, **disc_metrics(y_true,pred_proba[m]),
          **cal_metrics(y_true,pred_proba[m])} for m in names])

    pr,pauc,ppr=[],[],[]
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            a,b=names[i],names[j]
            da,dp=boot_delta_disc(y_true,pred_proba[a],pred_proba[b],n_boot,seed)
            la,ha=ci_from_boot(da); lp,hp=ci_from_boot(dp)
            pa_v,pp_v=two_sided_p(da),two_sided_p(dp)
            pauc.append(pa_v); ppr.append(pp_v)
            pr.append({"Compare":f"{b} - {a}",
                       "Delta_AUROC":float(np.nanmean(da)) if da.size else np.nan,
                       "Delta_AUROC_CI_low":la,"Delta_AUROC_CI_high":ha,
                       "p_AUROC":float(pa_v) if np.isfinite(pa_v) else np.nan,
                       "Delta_AUPRC":float(np.nanmean(dp)) if dp.size else np.nan,
                       "Delta_AUPRC_CI_low":lp,"Delta_AUPRC_CI_high":hp,
                       "p_AUPRC":float(pp_v) if np.isfinite(pp_v) else np.nan})
    pairwise=pd.DataFrame(pr)
    if len(pairwise):
        pairwise["q_AUROC_BH"]=bh_fdr(np.array(pauc,dtype=float))
        pairwise["q_AUPRC_BH"]=bh_fdr(np.array(ppr,dtype=float))

    # NRI pairs (원본과 동일)
    nri_pairs=[]
    if "base" in pred_proba and "base_lsm" in pred_proba: nri_pairs.append(("base","base_lsm","Within-B1"))
    if "base_lsm" in pred_proba and "base_lsm_cap" in pred_proba: nri_pairs.append(("base_lsm","base_lsm_cap","Within-B1"))
    if "base" in pred_proba and "base_lsm_cap" in pred_proba: nri_pairs.append(("base","base_lsm_cap","Within-B1"))
    if "base" in pred_proba and "base_inflam" in pred_proba: nri_pairs.append(("base","base_inflam","AST-ALT-standalone"))
    if "base_inflam" in pred_proba and "base_inflam_lsm" in pred_proba: nri_pairs.append(("base_inflam","base_inflam_lsm","Within-B2"))
    if "base_inflam_lsm" in pred_proba and "base_inflam_lsm_cap" in pred_proba: nri_pairs.append(("base_inflam_lsm","base_inflam_lsm_cap","Within-B2"))
    if "base_inflam" in pred_proba and "base_inflam_lsm_cap" in pred_proba: nri_pairs.append(("base_inflam","base_inflam_lsm_cap","Within-B2"))
    if "base_lsm" in pred_proba and "base_inflam_lsm" in pred_proba: nri_pairs.append(("base_lsm","base_inflam_lsm","Cross-block"))
    if "base_lsm_cap" in pred_proba and "base_inflam_lsm_cap" in pred_proba: nri_pairs.append(("base_lsm_cap","base_inflam_lsm_cap","Cross-block"))

    nr,pc_all,pk_all=[],[],[]
    for ref,new,grp in nri_pairs:
        dc,dk=boot_nri(y_true,pred_proba[ref],pred_proba[new],n_boot,seed)
        lc,hc=ci_from_boot(dc); lk,hk=ci_from_boot(dk)
        pc,pk=two_sided_p(dc),two_sided_p(dk)
        pc_all.append(pc); pk_all.append(pk)
        nr.append({"Group":grp, "Reference":ref, "Compare":f"{new} - {ref}",
                   "CatNRI":float(np.nanmean(dc)) if dc.size else np.nan,
                   "CatNRI_CI_low":lc,"CatNRI_CI_high":hc,
                   "p_CatNRI":float(pc) if np.isfinite(pc) else np.nan,
                   "ContNRI":float(np.nanmean(dk)) if dk.size else np.nan,
                   "ContNRI_CI_low":lk,"ContNRI_CI_high":hk,
                   "p_ContNRI":float(pk) if np.isfinite(pk) else np.nan})
    nri_tbl=None
    if nr:
        nri_tbl=pd.DataFrame(nr)
        nri_tbl["q_CatNRI_BH"]=bh_fdr(np.array(pc_all,dtype=float))
        nri_tbl["q_ContNRI_BH"]=bh_fdr(np.array(pk_all,dtype=float))

    # C-index
    st,se=build_surv(df_split,event_col,tte_col,fu_col,horizon_days)
    ci_rows=[]
    for k,m in enumerate(names):
        cp,lo,hi=boot_cindex(st,se,pred_proba[m],n_boot,seed+100+k)
        ci_rows.append({"Model":m,"Harrell_C":cp,"CI_low":lo,"CI_high":hi})
    cindex=pd.DataFrame(ci_rows)
    cindex["Harrell_C_95CI"]=cindex.apply(
        lambda r:f'{r.Harrell_C:.3f} ({r.CI_low:.3f}\u2013{r.CI_high:.3f})',axis=1)

    cp_rows,pc2=[],[]
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            a,b=names[i],names[j]
            d=boot_delta_cindex(st,se,pred_proba[a],pred_proba[b],n_boot,seed)
            lo,hi=ci_from_boot(d); p=two_sided_p(d); pc2.append(p)
            cp_rows.append({"Compare":f"{b} - {a}",
                            "Delta_Cindex":float(np.nanmean(d)) if d.size else np.nan,
                            "Delta_Cindex_CI_low":lo,"Delta_Cindex_CI_high":hi,
                            "p_Cindex":float(p) if np.isfinite(p) else np.nan})
    pairwise_c=pd.DataFrame(cp_rows)
    if len(pairwise_c):
        pairwise_c["q_Cindex_BH"]=bh_fdr(np.array(pc2,dtype=float))

    print("[EVAL] Done.")
    return dict(summary=summary, pairwise_discrimination=pairwise,
                nri=nri_tbl, cindex=cindex, pairwise_cindex=pairwise_c)


print("[CELL 8] Evaluation functions restored ✅")


[CELL 8] Evaluation functions restored ✅


## Cell 9 — `run_eval` 실행 → `res_A_test`, `res_A_ext`, `res_B_test`, `res_B_ext` 복원

**원본 Cell 28 재현.** `N_BOOT=1000`이라 좀 시간이 걸립니다(원본 대비 동일).

In [ ]:
def run_eval(label, y, pred, df_imp, event_col, tte_col, n_boot, seed):
    print()
    print("="*80); print(f"  {label}"); print("="*80)
    t0 = time.time()
    result = evaluate_models_full(
        y, pred, df_imp,
        event_col=event_col, tte_col=tte_col,
        n_boot=n_boot, seed=seed,
    )
    elapsed = time.time() - t0
    mins, secs = divmod(int(elapsed), 60)
    print(f"  ✅ Done in {mins}m {secs}s")
    return result


total_t0 = time.time()

res_A_test = run_eval("Scenario A — Internal test (CAP-era)",
    y_test_A, pred_test_A, df_test_A_imp,
    event_col=event_col, tte_col=tte_col, n_boot=N_BOOT, seed=RANDOM_STATE)

res_A_ext = run_eval("Scenario A — External validation (CAP-era)",
    y_ext_A, pred_ext_A, df_ext_A_imp,
    event_col=event_col, tte_col=tte_col, n_boot=N_BOOT, seed=RANDOM_STATE)

res_B_test = run_eval("Scenario B — Internal test (full+MICE+aux)",
    y_test, pred_test_B, df_test_B_imp,
    event_col=event_col, tte_col=tte_col, n_boot=N_BOOT, seed=RANDOM_STATE)

res_B_ext = run_eval("Scenario B — External validation (full+MICE+aux)",
    y_ext, pred_ext_B, df_ext_B_imp,
    event_col=event_col, tte_col=tte_col, n_boot=N_BOOT, seed=RANDOM_STATE)

total_elapsed = time.time() - total_t0
tm, ts = divmod(int(total_elapsed), 60)
print(f"\n🎉 All evaluations complete — total {tm}m {ts}s")



  Scenario A — Internal test (CAP-era)
[EVAL] models=['base', 'base_lsm', 'base_lsm_cap', 'base_inflam', 'base_inflam_lsm', 'base_inflam_lsm_cap']


/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid 

[EVAL] Done.
  ✅ Done in 117m 26s

  Scenario A — External validation (CAP-era)
[EVAL] models=['base', 'base_lsm', 'base_lsm_cap', 'base_inflam', 'base_inflam_lsm', 'base_inflam_lsm_cap']


/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid 

[EVAL] Done.
  ✅ Done in 249m 45s

  Scenario B — Internal test (full+MICE+aux)
[EVAL] models=['base', 'base_lsm', 'base_lsm_cap', 'base_inflam', 'base_inflam_lsm', 'base_inflam_lsm_cap']


/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid 

[EVAL] Done.
  ✅ Done in 325m 4s

  Scenario B — External validation (full+MICE+aux)
[EVAL] models=['base', 'base_lsm', 'base_lsm_cap', 'base_inflam', 'base_inflam_lsm', 'base_inflam_lsm_cap']


/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntu/tabpfn_gpu/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid 

## Cell 10 — Sanity check: 원본과 일치하는지 확인

저장 직전에 메모로 적어둔 값(예: 원본 `res_A_test['summary']`의 첫 번째 AUROC)과
비교하면 복구가 정확히 됐는지 확인할 수 있습니다.

In [ ]:
def show(label, res):
    print(f"\n{'='*80}\n  {label}\n{'='*80}")
    print("\n-- Discrimination & Calibration --"); display(res["summary"])
    print("\n-- Pairwise AUROC/AUPRC --");        display(res["pairwise_discrimination"])
    print("\n-- NRI (p-values + BH-FDR) --")
    display(res["nri"]) if res["nri"] is not None else print("  (n/a)")
    print("\n-- Harrell C-index --");              display(res["cindex"])
    print("\n-- Pairwise C-index --");             display(res["pairwise_cindex"])

show("Scenario A | Internal test",  res_A_test)
show("Scenario A | External valid", res_A_ext)
show("Scenario B | Internal test",  res_B_test)
show("Scenario B | External valid", res_B_ext)

print("\n" + "="*80)
print("✅ RECOVERY COMPLETE")
print("="*80)
print("이후 원본 노트북의 Cell 38 이하(calibration plots, SHAP, DCA)를 그대로 실행 가능.")
print("필요 변수 체크리스트:")
print("  - y_test_A, y_test, y_ext_A, y_ext            ✅")
print("  - df_test_A_imp, df_test_B_imp, df_ext_A_imp, df_ext_B_imp  ✅")
print("  - pred_test_A, pred_test_B, pred_ext_A, pred_ext_B          ✅")
print("  - models_A, models_B, imp_A, imp_B            ✅ (SHAP용)")
print("  - res_A_test, res_A_ext, res_B_test, res_B_ext ✅")
